# EM Few-shot Classification — 학습 → 테스트 데모

frozen DINO(teacher) feature 위에 가벼운 헤드(logreg/NCM)를 얹어 EM 이미지를 분류한다.
백본은 건드리지 않고 feature 추출기로만 쓴다.

**흐름**: 백본 로드 → feature 추출 → 헤드 학습/검증 → 아티팩트 저장 → 로드 → 추론 → heatmap → 테스트폴더 일괄 처리

> ⚠️ 서버(torch+CUDA)에서 실행. 아래 **설정** 셀의 경로를 채운 뒤 위에서부터 실행.
> 학습이 `instance_norm`(PerImageNormalize)이라 feature 추출도 동일 정규화를 쓴다(도구 내장).

## 0. 경로/임포트

In [ ]:
import os, sys
from pathlib import Path

# repo 루트 + dino_v3 를 path 에 (노트북 위치: tests/classification/)
_HERE = Path.cwd()
_REPO = _HERE
while not (_REPO / 'dino_v3').exists() and _REPO != _REPO.parent:
    _REPO = _REPO.parent
for p in (str(_REPO), str(_REPO / 'dino_v3')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('repo:', _REPO)

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torchvision import datasets

from dinov3.eval.em_classifier import (
    EMFeatureExtractor, ClassifierHead, EMClassifier, fit_from_imagefolder,
)
from dinov3.eval.fewshot_separability import extract_features
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. 설정 (여기만 채우면 됨)

- `CONFIG`: 학습에 쓴 config yaml (arch 일치용)
- `CKPT`: teacher 체크포인트(.pth 또는 DCP 디렉토리)
- `DATA_ROOT`: 클래스별 폴더(ImageFolder)

In [ ]:
CONFIG    = 'dino_v3/dinov3/configs/train/weaksup/stage2_ssl_weaksup.yaml'  # FILL_IN
CKPT      = '/path/to/teacher_checkpoint.pth'   # FILL_IN
DATA_ROOT = '/path/to/class_folders'            # FILL_IN  (classA/ classB/ ...)
OUT       = './out/em_head.npz'                 # 아티팩트 저장 경로

IMAGE_SIZE = 448      # 학습 global crop 과 동일 권장
FEATURE    = 'concat' # cls | patchmean | concat
N_BLOCKS   = 1
CLF        = 'logreg' # logreg | ncm
VAL_FRAC   = 0.3      # 검증 비율
SEED       = 0
os.makedirs(Path(OUT).parent, exist_ok=True)
CONFIG = str((_REPO / CONFIG)) if not os.path.isabs(CONFIG) else CONFIG

## 2. 백본 로드 (1회, 무거움)
분산은 단일 프로세스로 자동 초기화된다(torchrun 불필요).

In [ ]:
extractor = EMFeatureExtractor(
    CONFIG, CKPT, image_size=IMAGE_SIZE, n_blocks=N_BLOCKS,
    feature_kind=FEATURE, cache_dir='./.em_cache',
)
print('embed_dim:', extractor.embed_dim, '| feature:', extractor.feature_kind)

## 3. feature 추출 + train/val split
전체 이미지를 한 번 forward 해서 feature 로 변환(여기까지가 백본 사용의 전부).

In [ ]:
dataset = datasets.ImageFolder(DATA_ROOT, transform=extractor._transform())
class_names = dataset.classes
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
X, y = extract_features(extractor.model, loader, N_BLOCKS, FEATURE, extractor.autocast_dtype)
print('feature:', X.shape, '| classes:', class_names)
for c, n in zip(class_names, np.bincount(y, minlength=len(class_names))):
    print(f'  {c}: {n}')

from sklearn.model_selection import train_test_split
tr, va = train_test_split(np.arange(len(y)), test_size=VAL_FRAC, stratify=y, random_state=SEED)
print('train/val:', len(tr), len(va))

## 4. 헤드 학습 + 검증
train split 으로 헤드를 학습하고 val 에서 정확도/혼동행렬 확인.

In [ ]:
head_eval = ClassifierHead.fit(X[tr], y[tr], class_names, kind=CLF)
pred = head_eval.predict(X[va])
acc = (pred == y[va]).mean()
print(f'val accuracy: {acc*100:.1f}%')

# 혼동행렬
C = len(class_names)
cm = np.zeros((C, C), int)
for p, t in zip(pred, y[va]):
    cm[t, p] += 1
fig, ax = plt.subplots(figsize=(1.2*C+2, 1.2*C+1))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(C)); ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticks(range(C)); ax.set_yticklabels(class_names)
ax.set_xlabel('pred'); ax.set_ylabel('true'); ax.set_title(f'confusion (val acc {acc*100:.1f}%)')
for i in range(C):
    for j in range(C):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.tight_layout(); plt.show()

## 5. 최종 헤드 학습(전체 데이터) + 아티팩트 저장
검증으로 성능을 확인했으니, 배포용은 전체 데이터로 다시 학습해 저장한다.
아티팩트(.npz)에 W/bias/클래스명 + feature 설정 + 백본 경로가 함께 들어간다.

In [ ]:
head = fit_from_imagefolder(extractor, DATA_ROOT, kind=CLF, num_workers=4)
path = head.save(OUT)
print('saved:', path)
print('meta:', head.meta)

## 6. 로드 + 추론
실제 서비스라면 `EMClassifier.from_artifact(OUT)` 로 백본까지 새로 로드하지만,
여기선 이미 로드된 백본을 재사용한다.

In [ ]:
loaded = ClassifierHead.load(OUT)
clf = EMClassifier(extractor, loaded)   # 백본 재사용

# val 에서 몇 장 골라 추론 결과 시각화
samples = [dataset.samples[i] for i in va[:6]]
imgs = [Image.open(p).convert('RGB') for p, _ in samples]
res = clf.predict(imgs, topk=min(3, len(class_names)))

n = len(imgs)
fig, axes = plt.subplots(1, n, figsize=(3*n, 3.5))
if n == 1: axes = [axes]
for ax, im, (p, t), r in zip(axes, imgs, samples, res):
    ax.imshow(im.resize((IMAGE_SIZE, IMAGE_SIZE)), cmap='gray')
    ok = (r['label'] == class_names[t])
    ax.set_title(f"pred {r['label']} ({r['score']:.2f})\ntrue {class_names[t]}", color=('green' if ok else 'red'), fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 7. 헬퍼: class-evidence heatmap (contrastive)

`W_patch·patch_p` 를 그대로 쓰면 **공통 DC 성분** 때문에 거의 전부 빨갛게 나온다.
→ **contrastive**: `W_patch[pred] − mean(W_patch[others])` 로 다른 클래스 대비 판별 신호만 본다.
스케일은 99 퍼센타일로 robust 하게 (아웃라이어 무시). 빨강=이 클래스 지지, 파랑=반대.

In [ ]:
def class_evidence_heatmap(extractor, head, pil, image_size, n_blocks, target=None, mode='contrastive'):
    """returns (heat[image_size,image_size], pred:int, proba[num_classes])."""
    embed_dim = extractor.embed_dim
    x = extractor._transform()(pil.convert('RGB')).unsqueeze(0).cuda()
    with torch.inference_mode(), torch.autocast('cuda', dtype=extractor.autocast_dtype):
        outs = extractor.model.get_intermediate_layers(x, n=n_blocks, reshape=True, return_class_token=True, norm=True)
    patch_grid = outs[-1][0][0].float().cpu().numpy()   # [C,h,w]
    proba = head.predict_proba(extractor.features(pil))[0]
    pred = int(proba.argmax()) if target is None else int(target)
    Wp = head.W[:, -embed_dim:]                          # [num_classes, C]
    if mode == 'contrastive' and head.num_classes > 1:
        others = np.delete(np.arange(head.num_classes), pred)
        w = Wp[pred] - Wp[others].mean(axis=0)
    else:  # raw: DC 제거만
        w = Wp[pred]
    heat = np.einsum('chw,c->hw', patch_grid, w)
    if mode == 'raw':
        heat = heat - heat.mean()
    heat = torch.nn.functional.interpolate(
        torch.from_numpy(heat)[None, None].float(), size=(image_size, image_size),
        mode='bilinear', align_corners=False)[0, 0].numpy()
    return heat, pred, proba

def save_overlay(pil, heat, out_path, title, image_size, alpha=0.5):
    img = np.asarray(pil.resize((image_size, image_size)).convert('L'), float) / 255
    vmax = float(np.percentile(np.abs(heat), 99)) + 1e-8   # robust
    fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
    ax[0].imshow(img, cmap='gray'); ax[0].set_title('input'); ax[0].axis('off')
    ax[1].imshow(img, cmap='gray')
    hm = ax[1].imshow(heat, cmap='seismic', vmin=-vmax, vmax=vmax, alpha=alpha)
    ax[1].set_title(title); ax[1].axis('off')
    fig.colorbar(hm, ax=ax[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=130, bbox_inches='tight')
    return fig

## 8. heatmap 한 장 확인
contrastive vs raw 비교해 보면 판별 영역이 더 또렷한지 알 수 있다.

In [ ]:
assert FEATURE == 'concat', 'heatmap 은 concat feature 필요'
pil = Image.open(samples[0][0]).convert('RGB')
for mode in ['contrastive', 'raw']:
    heat, pred, proba = class_evidence_heatmap(extractor, loaded, pil, IMAGE_SIZE, N_BLOCKS, mode=mode)
    save_overlay(pil, heat, None, f'[{mode}] {class_names[pred]} ({proba[pred]:.2f})', IMAGE_SIZE)
    plt.show()

## 9. 테스트 폴더 일괄 처리 — 판정 결과 + heatmap 저장

`TEST_DIR` 안의 모든 이미지에 대해:
1. 클래스 판정 결과를 `predictions.csv` / `predictions.json` 으로 저장
2. 각 이미지의 heatmap overlay 를 `heatmaps/` 에 PNG 로 저장

In [ ]:
TEST_DIR   = '/path/to/test_images'   # FILL_IN (라벨 없어도 됨)
RESULT_DIR = './out/test_results'
HEAT_MODE  = 'contrastive'            # contrastive | raw
TOPK       = min(3, len(class_names))

import csv, json
heat_dir = os.path.join(RESULT_DIR, 'heatmaps')
os.makedirs(heat_dir, exist_ok=True)
exts = {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}
paths = [p for p in sorted(Path(TEST_DIR).rglob('*')) if p.suffix.lower() in exts]
print(f'{len(paths)} images in {TEST_DIR}')

rows = []
for i, p in enumerate(paths):
    pil = Image.open(p).convert('RGB')
    r = clf.predict(pil, topk=TOPK)
    heat, pred, proba = class_evidence_heatmap(extractor, loaded, pil, IMAGE_SIZE, N_BLOCKS, mode=HEAT_MODE)
    title = f"{r['label']} ({r['score']:.2f})"
    fig = save_overlay(pil, heat, os.path.join(heat_dir, f'{p.stem}_heat.png'), title, IMAGE_SIZE)
    plt.close(fig)
    rows.append({'file': p.name, 'pred': r['label'], 'score': round(r['score'], 4),
                 'topk': [(n, round(s, 4)) for n, s in r['topk']]})
    if (i + 1) % 20 == 0:
        print(f'  {i+1}/{len(paths)}')

with open(os.path.join(RESULT_DIR, 'predictions.json'), 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
with open(os.path.join(RESULT_DIR, 'predictions.csv'), 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f); w.writerow(['file', 'pred', 'score'])
    for r in rows:
        w.writerow([r['file'], r['pred'], r['score']])
print(f'완료 → {RESULT_DIR}  (predictions.csv/json, heatmaps/{len(rows)}장)')

In [ ]:
# 결과 미리보기 (있으면 표로)
try:
    import pandas as pd
    df = pd.DataFrame([{'file': r['file'], 'pred': r['pred'], 'score': r['score']} for r in rows])
    display(df.head(20))
    print('클래스별 예측 분포:'); print(df['pred'].value_counts())
except Exception:
    for r in rows[:20]:
        print(r['file'], '->', r['pred'], r['score'])

## 10. 단위 테스트
헤드 로직(numpy)은 torch 없이도 검증된다. `pytest tests/classification`.

In [ ]:
import subprocess
r = subprocess.run([sys.executable, '-m', 'pytest', str(_REPO / 'tests/classification'), '-q'],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-1000:])